# Relaxationszeiten aus exponentiellen Verlaeufen

Dieses Notebook bestimmt Zeitkonstanten aus den zeitlichen Verlaeufen von

- der direkt bestimmten Stufenhoehe `delta epsilon'` (`Eps_real_Step_Height`),
- der aus dem Fit bestimmten dielektrischen Staerke `delta epsilon` (`de`),
- der Peakposition `omega_p`.

Gefittet wird pro Material, Temperatur, Modus und Observable ein einfacher exponentieller Verlauf

`y(t) = y_inf + A * exp(-t / tau)`

wobei `tau` die gesuchte Relaxationszeit ist.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
PLOT_DIR = RESULTS_DIR / "relaxation_time_plots"

PROJECT_ROOT, RESULTS_DIR

## Einstellungen

`FIT_AFTER_SWITCH_ONLY = True` verwendet nur Messpunkte ab dem Schaltpunkt (`Time_Relative_s >= 0`). Wenn du den gesamten Verlauf fitten moechtest, setze den Wert auf `False`.

In [ ]:
USE_PILOT_FITS = False
FIT_AFTER_SWITCH_ONLY = True
MIN_TIME_S = 0.0
MIN_POINTS_PER_FIT = 6

# Optional einschraenken, z.B. TEMPERATURES_C = [50, 60, 70]
MATERIALS = None
TEMPERATURES_C = None
MODES = None

FIT_FILE_PATTERN = "fit_parameters_pilot_*.csv" if USE_PILOT_FITS else "fit_parameters_*.csv ohne pilot"
FIT_FILE_PATTERN

In [ ]:
def parse_temperature_c(value):
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan


def parse_fit_filename(path):
    match = re.match(
        r"fit_parameters(?:_pilot)?_(?P<material>.+?)_(?P<temperature>\d+)C_(?P<mode>Abs|Des)\.csv$",
        path.name,
    )
    if not match:
        raise ValueError(f"Dateiname passt nicht zum erwarteten Muster: {path.name}")
    return {
        "Material": match.group("material"),
        "Temperature_C": float(match.group("temperature")),
        "Mode": match.group("mode"),
    }


def load_direct_step_heights():
    all_series = RESULTS_DIR / "step_height_eps_real_all_series.csv"
    if all_series.exists():
        df = pd.read_csv(all_series)
    else:
        files = sorted(RESULTS_DIR.glob("step_height_eps_real_*C_*.csv"))
        df = pd.concat((pd.read_csv(path) for path in files), ignore_index=True)

    df = df.copy()
    df["Temperature_C"] = df["Temperature"].map(parse_temperature_c)
    return df


def load_fit_parameters():
    if USE_PILOT_FITS:
        files = sorted(RESULTS_DIR.glob("fit_parameters_pilot_*.csv"))
    else:
        files = sorted(
            path for path in RESULTS_DIR.glob("fit_parameters_*.csv")
            if not path.name.startswith("fit_parameters_pilot_")
        )
    if not files:
        raise FileNotFoundError(f"Keine Fitparameter-Dateien gefunden: {FIT_FILE_PATTERN}")

    frames = []
    for path in files:
        df = pd.read_csv(path)
        for key, value in parse_fit_filename(path).items():
            df[key] = value
        df["Source_File"] = path.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


direct_df = load_direct_step_heights()
fit_df = load_fit_parameters()

direct_df.head(), fit_df.head()

In [ ]:
def apply_selection(df):
    selected = df.copy()
    if MATERIALS is not None:
        selected = selected[selected["Material"].isin(MATERIALS)]
    if TEMPERATURES_C is not None:
        selected = selected[selected["Temperature_C"].isin(TEMPERATURES_C)]
    if MODES is not None:
        selected = selected[selected["Mode"].isin(MODES)]
    return selected


def to_observable_frame(df, value_column, observable, source):
    required = ["Material", "Temperature_C", "Mode", "Time_Relative_s", value_column]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Fehlende Spalten fuer {observable}: {missing}")

    out = df[required].copy()
    out = out.rename(columns={value_column: "value"})
    out["observable"] = observable
    out["source"] = source
    return out


direct_selected = apply_selection(direct_df)
fit_selected = apply_selection(fit_df)

if "Accepted" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Accepted"].fillna(False).astype(bool)]
if "Fit_Success" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Fit_Success"].fillna(False).astype(bool)]

observables = [
    to_observable_frame(direct_selected, "Eps_real_Step_Height", "delta_eps_real_step_height", "step_height"),
    to_observable_frame(fit_selected, "de", "delta_eps_fit_de", "fit"),
    to_observable_frame(fit_selected, "omega_p", "omega_p", "fit"),
]

data_long = pd.concat(observables, ignore_index=True)
data_long = data_long.replace([np.inf, -np.inf], np.nan).dropna(subset=["Time_Relative_s", "value"])
data_long = data_long.sort_values(["Material", "Temperature_C", "Mode", "observable", "Time_Relative_s"])

data_long.groupby(["Material", "Temperature_C", "Mode", "observable"]).size().rename("n_points").reset_index()

In [ ]:
def exp_model(t, y_inf, amplitude, tau):
    return y_inf + amplitude * np.exp(-t / tau)


def fit_exponential(group):
    group = group.sort_values("Time_Relative_s").copy()
    if FIT_AFTER_SWITCH_ONLY:
        group = group[group["Time_Relative_s"] >= MIN_TIME_S]

    group = group.dropna(subset=["Time_Relative_s", "value"])
    if len(group) < MIN_POINTS_PER_FIT:
        return None, group, "too_few_points"

    t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    t = t_raw - np.nanmin(t_raw)

    if np.nanmax(t) <= 0 or np.nanstd(y) == 0:
        return None, group, "not_enough_variation"

    tail_n = max(3, len(y) // 5)
    y_inf0 = float(np.nanmedian(y[-tail_n:]))
    amplitude0 = float(y[0] - y_inf0)
    if amplitude0 == 0:
        amplitude0 = float(np.nanmax(y) - np.nanmin(y))
    tau0 = max(float(np.nanmax(t) / 3), 1e-9)

    y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-12)
    lower = [float(np.nanmin(y) - 5 * y_span), -10 * y_span, 1e-9]
    upper = [float(np.nanmax(y) + 5 * y_span), 10 * y_span, max(float(np.nanmax(t) * 100), 1.0)]

    try:
        popt, pcov = curve_fit(
            exp_model,
            t,
            y,
            p0=[y_inf0, amplitude0, tau0],
            bounds=(lower, upper),
            maxfev=20000,
        )
    except Exception as exc:
        return None, group, f"fit_failed: {exc}"

    y_fit = exp_model(t, *popt)
    residuals = y - y_fit
    ss_res = float(np.sum(residuals**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean(residuals**2)))

    perr = np.full(3, np.nan)
    if pcov is not None and np.all(np.isfinite(pcov)):
        perr = np.sqrt(np.diag(pcov))

    result = {
        "n_points": len(group),
        "time_start_s": float(np.nanmin(t_raw)),
        "time_end_s": float(np.nanmax(t_raw)),
        "y_inf": float(popt[0]),
        "amplitude": float(popt[1]),
        "tau_s": float(popt[2]),
        "tau_err_s": float(perr[2]) if np.isfinite(perr[2]) else np.nan,
        "rmse": rmse,
        "r2": float(r2) if np.isfinite(r2) else np.nan,
        "fit_status": "ok",
    }
    return result, group.assign(t_fit_s=t, y_fit=y_fit), "ok"


fit_rows = []
fit_curves = []
group_columns = ["Material", "Temperature_C", "Mode", "observable"]

for group_key, group in data_long.groupby(group_columns):
    result, curve, status = fit_exponential(group)
    row = dict(zip(group_columns, group_key))
    if result is None:
        row.update({"fit_status": status, "n_points": len(curve), "tau_s": np.nan, "tau_err_s": np.nan, "r2": np.nan, "rmse": np.nan})
    else:
        row.update(result)
        fit_curves.append(curve.assign(**row))
    fit_rows.append(row)

relaxation_results = pd.DataFrame(fit_rows).sort_values(group_columns)
relaxation_results

In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def plot_group(group, result_row, save=False):
    material = result_row["Material"]
    temp = result_row["Temperature_C"]
    mode = result_row["Mode"]
    observable = result_row["observable"]

    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.scatter(group["Time_Relative_s"], group["value"], s=18, label="Daten")

    if result_row.get("fit_status") == "ok":
        t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
        t_plot_raw = np.linspace(np.nanmin(t_raw), np.nanmax(t_raw), 300)
        t_plot = t_plot_raw - np.nanmin(t_raw)
        y_plot = exp_model(t_plot, result_row["y_inf"], result_row["amplitude"], result_row["tau_s"])
        ax.plot(t_plot_raw, y_plot, color="tab:red", label=f"Fit: tau = {result_row['tau_s']:.3g} s")

    ax.set_title(f"{material}, {temp:g} C, {mode}: {observable}")
    ax.set_xlabel("relative Zeit / s")
    ax.set_ylabel(observable)
    ax.legend()
    fig.tight_layout()

    if save:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        filename = f"relaxation_{safe_name(material)}_{temp:g}C_{mode}_{safe_name(observable)}.png"
        fig.savefig(PLOT_DIR / filename, bbox_inches="tight")
    return fig, ax


with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        mask = (
            (data_long["Material"] == result_row["Material"])
            & (data_long["Temperature_C"] == result_row["Temperature_C"])
            & (data_long["Mode"] == result_row["Mode"])
            & (data_long["observable"] == result_row["observable"])
        )
        group = data_long.loc[mask].sort_values("Time_Relative_s")
        if FIT_AFTER_SWITCH_ONLY:
            group = group[group["Time_Relative_s"] >= MIN_TIME_S]
        plot_group(group, result_row, save=False)
        plt.show()

print("Es wurde nichts gespeichert. Zum Speichern die letzte Zelle verwenden.")

In [ ]:
summary = relaxation_results.pivot_table(
    index=["Material", "Temperature_C", "Mode"],
    columns="observable",
    values="tau_s",
    aggfunc="first",
)
summary

## Optional speichern

Erst wenn die Ergebnisse plausibel aussehen, `SPEICHERN = True` setzen und diese Zelle ausfuehren.

In [ ]:
SPEICHERN = False

if SPEICHERN:
    output_csv = RESULTS_DIR / "relaxation_times_exponential_fits.csv"
    relaxation_results.to_csv(output_csv, index=False)

    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    saved_plots = 0
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        mask = (
            (data_long["Material"] == result_row["Material"])
            & (data_long["Temperature_C"] == result_row["Temperature_C"])
            & (data_long["Mode"] == result_row["Mode"])
            & (data_long["observable"] == result_row["observable"])
        )
        group = data_long.loc[mask].sort_values("Time_Relative_s")
        if FIT_AFTER_SWITCH_ONLY:
            group = group[group["Time_Relative_s"] >= MIN_TIME_S]
        fig, _ = plot_group(group, result_row, save=True)
        plt.close(fig)
        saved_plots += 1

    print(f"Gespeichert: {output_csv}")
    print(f"Gespeicherte Plots: {saved_plots} in {PLOT_DIR}")
else:
    print("Nichts gespeichert. Setze SPEICHERN = True, wenn du die Ergebnisse exportieren willst.")